In [29]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [30]:
import os

base_dir = "/content/drive/MyDrive/iot/iot device name"

pcap2 = os.path.join(base_dir, "normal_IoT_2.pcap")
pcap3 = os.path.join(base_dir, "normal_IoT_3.pcap")

print(os.path.exists(pcap2), pcap2)
print(os.path.exists(pcap3), pcap3)

True /content/drive/MyDrive/iot/iot device name/normal_IoT_2.pcap
True /content/drive/MyDrive/iot/iot device name/normal_IoT_3.pcap


In [31]:
import pandas as pd
from collections import Counter
from scapy.all import PcapReader, CookedLinux, IP, ARP, TCP, UDP


def is_local_ip(ip_addr):
    if ip_addr is None:
        return False

    ip_text = str(ip_addr)

    if ip_text.startswith("192.168."):
        return True
    if ip_text.startswith("10."):
        return True

    for i in range(16, 32):
        if ip_text.startswith(f"172.{i}."):
            return True

    return False


def format_mac(raw_mac):
    if raw_mac is None:
        return None

    if isinstance(raw_mac, bytes):
        return ":".join(f"{b:02x}" for b in raw_mac[:6])

    if isinstance(raw_mac, str):
        return raw_mac

    return None


def collect_arp_pairs(pcap_file):
    arp_counter = Counter()

    with PcapReader(pcap_file) as packets:
        for pkt in packets:
            if ARP not in pkt:
                continue

            try:
                src_mac = pkt[ARP].hwsrc
                src_ip = pkt[ARP].psrc

                if src_mac and src_ip and src_ip != "0.0.0.0":
                    arp_counter[(src_mac, src_ip)] += 1

            except Exception:
                continue

    rows = []
    for (mac_addr, ip_addr), seen_count in arp_counter.items():
        rows.append({
            "mac": mac_addr,
            "ip": ip_addr,
            "count": seen_count
        })

    arp_df = pd.DataFrame(rows)

    if not arp_df.empty:
        arp_df = arp_df.sort_values("count", ascending=False).reset_index(drop=True)

    return arp_df


def collect_sll_pairs(pcap_file):
    sll_counter = Counter()

    with PcapReader(pcap_file) as packets:
        for pkt in packets:
            if CookedLinux not in pkt or IP not in pkt:
                continue

            try:
                sll_layer = pkt[CookedLinux]

                mac_addr = format_mac(getattr(sll_layer, "src", None))
                ip_addr = pkt[IP].src
                addr_type = getattr(sll_layer, "lladdrtype", None)
                addr_len = getattr(sll_layer, "lladdrlen", None)

                has_valid_mac = mac_addr is not None and mac_addr != "00:00:00:00:00:00"
                has_valid_ip = is_local_ip(ip_addr)

                if addr_type == 1 and addr_len == 6 and has_valid_mac and has_valid_ip:
                    sll_counter[(mac_addr, ip_addr)] += 1

            except Exception:
                continue

    rows = []
    for (mac_addr, ip_addr), seen_count in sll_counter.items():
        rows.append({
            "mac": mac_addr,
            "ip": ip_addr,
            "count": seen_count
        })

    sll_df = pd.DataFrame(rows)

    if not sll_df.empty:
        sll_df = sll_df.sort_values("count", ascending=False).reset_index(drop=True)

    return sll_df


def count_ip_flows(pcap_file):
    flow_counter = Counter()

    with PcapReader(pcap_file) as packets:
        for pkt in packets:
            if IP not in pkt:
                continue

            src_ip = pkt[IP].src
            dst_ip = pkt[IP].dst

            src_port = None
            dst_port = None
            proto_name = str(pkt[IP].proto)

            if TCP in pkt:
                src_port = int(pkt[TCP].sport)
                dst_port = int(pkt[TCP].dport)
                proto_name = "TCP"
            elif UDP in pkt:
                src_port = int(pkt[UDP].sport)
                dst_port = int(pkt[UDP].dport)
                proto_name = "UDP"

            flow_key = (src_ip, dst_ip, src_port, dst_port, proto_name)
            flow_counter[flow_key] += 1

    rows = []
    for flow_key, packet_total in flow_counter.items():
        rows.append({
            "src_ip": flow_key[0],
            "dst_ip": flow_key[1],
            "src_port": flow_key[2],
            "dst_port": flow_key[3],
            "proto": flow_key[4],
            "packet_count": packet_total
        })

    flow_df = pd.DataFrame(rows)

    if not flow_df.empty:
        flow_df = flow_df.sort_values("packet_count", ascending=False).reset_index(drop=True)

    return flow_df


def get_confidence_level(row):
    if row["num_macs"] == 1:
        return "high"

    if row["best_ratio"] >= 0.85:
        return "medium"

    return "low"


def add_device_labels(flow_df, ip_to_device, ip_to_mac):
    labeled_df = flow_df.copy()

    def choose_local_ip(row):
        if is_local_ip(row["src_ip"]):
            return row["src_ip"]

        if is_local_ip(row["dst_ip"]):
            return row["dst_ip"]

        return None

    labeled_df["local_ip"] = labeled_df.apply(choose_local_ip, axis=1)
    labeled_df["device_name"] = labeled_df["local_ip"].map(ip_to_device)
    labeled_df["device_mac"] = labeled_df["local_ip"].map(ip_to_mac)

    def judge_direction(row):
        if pd.isna(row["local_ip"]):
            return "nonlocal"

        if row["src_ip"] == row["local_ip"]:
            return "out"

        return "in"

    labeled_df["direction"] = labeled_df.apply(judge_direction, axis=1)
    labeled_df["is_labeled"] = labeled_df["device_name"].notna()

    return labeled_df


def add_service_labels(flow_df, ip_to_service, ip_to_mac):
    labeled_df = flow_df.copy()

    def choose_local_ip(row):
        if is_local_ip(row["src_ip"]):
            return row["src_ip"]

        if is_local_ip(row["dst_ip"]):
            return row["dst_ip"]

        return None

    labeled_df["local_ip"] = labeled_df.apply(choose_local_ip, axis=1)
    labeled_df["service_name"] = labeled_df["local_ip"].map(ip_to_service)
    labeled_df["service_mac"] = labeled_df["local_ip"].map(ip_to_mac)

    def judge_direction(row):
        if pd.isna(row["local_ip"]):
            return "nonlocal"

        if row["src_ip"] == row["local_ip"]:
            return "out"

        return "in"

    labeled_df["direction"] = labeled_df.apply(judge_direction, axis=1)
    labeled_df["is_labeled"] = labeled_df["service_name"].notna()

    return labeled_df

In [32]:
arp_map_2 = collect_arp_pairs(pcap2)
arp_map_3 = collect_arp_pairs(pcap3)

print("=== arp_map_2 ===")
display(arp_map_2.head(20))

print("=== arp_map_3 ===")
display(arp_map_3.head(20))


sll_map_2 = collect_sll_pairs(pcap2)
sll_map_3 = collect_sll_pairs(pcap3)

print("=== sll_map_2 ===")
display(sll_map_2.head(20))

print("=== sll_map_3 ===")
display(sll_map_3.head(20))

=== arp_map_2 ===


,mac,ip,count
0,a4:91:b1:1e:57:90,192.168.1.1,21744
1,00:c3:f4:0f:67:73,192.168.1.79,11060
2,00:0c:29:d2:b0:02,192.168.1.152,8961
3,00:0c:29:ee:e0:7a,192.168.1.190,4118
4,00:0c:29:a8:3a:da,192.168.1.195,1787
5,60:14:b3:b1:91:73,192.168.1.193,459
6,d4:dc:cd:b4:26:3e,192.168.1.250,67
7,dc:56:e7:5b:61:41,192.168.1.133,20
8,80:3f:5d:10:17:e1,192.168.1.17,17
9,9c:b6:d0:8e:86:79,192.168.1.103,4


=== arp_map_3 ===


,mac,ip,count
0,a4:91:b1:1e:57:90,192.168.1.1,23858
1,00:c3:f4:0f:67:73,192.168.1.79,12555
2,00:0c:29:d2:b0:02,192.168.1.152,11754
3,60:14:b3:b1:91:73,192.168.1.193,4490
4,00:0c:29:ee:e0:7a,192.168.1.190,3903
5,60:14:b3:b1:91:73,192.168.1.192,2152
6,00:0c:29:a8:3a:da,192.168.1.195,1516
7,60:14:b3:b1:91:73,192.168.1.194,1163
8,00:0c:29:7d:af:c3,192.168.1.195,366
9,80:3f:5d:10:17:e1,192.168.1.17,112


=== sll_map_2 ===


,mac,ip,count
0,00:0c:29:d2:b0:02,192.168.1.152,757967
1,00:0c:29:a8:3a:da,192.168.1.195,368712
2,00:0c:29:ee:e0:7a,192.168.1.190,11877
3,00:c3:f4:0f:67:73,192.168.1.79,3818
4,d4:dc:cd:b4:26:3e,192.168.1.250,1098
5,dc:56:e7:5b:61:41,192.168.1.133,875
6,a4:91:b1:1e:57:90,192.168.1.1,696
7,9c:b6:d0:8e:86:79,192.168.1.103,592
8,ec:1f:72:f1:28:6d,192.168.1.6,381
9,60:14:b3:b1:91:73,192.168.1.46,257


=== sll_map_3 ===


,mac,ip,count
0,00:0c:29:d2:b0:02,192.168.1.152,931658
1,00:0c:29:a8:3a:da,192.168.1.195,296249
2,60:14:b3:b1:91:73,192.168.1.192,122931
3,00:0c:29:7d:af:c3,192.168.1.195,72535
4,00:0c:29:ee:e0:7a,192.168.1.190,63550
5,60:14:b3:b1:91:73,192.168.1.193,52611
6,00:c3:f4:0f:67:73,192.168.1.79,4316
7,80:3f:5d:10:17:e1,192.168.1.30,2569
8,dc:56:e7:5b:61:41,192.168.1.133,1149
9,d4:dc:cd:b4:26:3e,192.168.1.250,904


In [33]:
arp2 = arp_map_2.copy()
arp3 = arp_map_3.copy()
sll2 = sll_map_2.copy()
sll3 = sll_map_3.copy()

arp2["source"] = "arp_2"
arp3["source"] = "arp_3"
sll2["source"] = "sll_2"
sll3["source"] = "sll_3"

all_maps = pd.concat(
    [
        arp2[["ip", "mac", "count", "source"]],
        arp3[["ip", "mac", "count", "source"]],
        sll2[["ip", "mac", "count", "source"]],
        sll3[["ip", "mac", "count", "source"]],
    ],
    ignore_index=True
)

ip_mac_counts = (
    all_maps.groupby(["ip", "mac"], as_index=False)["count"]
    .sum()
    .sort_values(["ip", "count"], ascending=[True, False])
)

ip_mac_count_df = (
    ip_mac_counts.groupby("ip")["mac"]
    .nunique()
    .reset_index(name="num_macs")
)

main_mac_df = (
    ip_mac_counts.drop_duplicates("ip")
    .rename(columns={"mac": "best_mac", "count": "best_count"})
)

ip_total_df = (
    ip_mac_counts.groupby("ip")["count"]
    .sum()
    .reset_index(name="total_count")
)

device_map_df = (
    main_mac_df.merge(ip_mac_count_df, on="ip", how="left")
    .merge(ip_total_df, on="ip", how="left")
)

device_map_df["best_ratio"] = device_map_df["best_count"] / device_map_df["total_count"]
device_map_df["confidence"] = device_map_df.apply(get_confidence_level, axis=1)

device_map_df = device_map_df.sort_values(
    ["confidence", "total_count"],
    ascending=[True, False]
).reset_index(drop=True)

display(device_map_df.head(30))

,ip,best_mac,best_count,num_macs,total_count,best_ratio,confidence
0,192.168.1.152,00:0c:29:d2:b0:02,1710340,1,1710340,1.00000,high
1,192.168.1.192,60:14:b3:b1:91:73,125095,1,125095,1.00000,high
2,192.168.1.190,00:0c:29:ee:e0:7a,83448,1,83448,1.00000,high
3,192.168.1.193,60:14:b3:b1:91:73,57810,1,57810,1.00000,high
4,192.168.1.1,a4:91:b1:1e:57:90,46868,1,46868,1.00000,high
5,192.168.1.79,00:c3:f4:0f:67:73,31749,1,31749,1.00000,high
6,192.168.1.30,80:3f:5d:10:17:e1,2581,1,2581,1.00000,high
7,192.168.1.250,d4:dc:cd:b4:26:3e,2101,1,2101,1.00000,high
8,192.168.1.133,dc:56:e7:5b:61:41,2056,1,2056,1.00000,high
9,192.168.1.194,60:14:b3:b1:91:73,1334,1,1334,1.00000,high


In [34]:
mac_summary_df = (
    ip_mac_counts.groupby("mac")
    .agg(
        total_count=("count", "sum"),
        num_ips=("ip", "nunique"),
        ip_list=("ip", lambda s: "; ".join(sorted(s.unique())))
    )
    .reset_index()
    .sort_values(["total_count"], ascending=False)
    .reset_index(drop=True)
)

display(mac_summary_df.head(30))

,mac,total_count,num_ips,ip_list
0,00:0c:29:d2:b0:02,1710340,1,192.168.1.152
1,00:0c:29:a8:3a:da,668264,1,192.168.1.195
2,60:14:b3:b1:91:73,184708,4,192.168.1.192; 192.168.1.193; 192.168.1.194; 1...
3,00:0c:29:ee:e0:7a,83448,1,192.168.1.190
4,00:0c:29:7d:af:c3,72901,1,192.168.1.195
5,a4:91:b1:1e:57:90,46868,1,192.168.1.1
6,00:c3:f4:0f:67:73,31749,1,192.168.1.79
7,80:3f:5d:10:17:e1,3349,3,192.168.1.17; 192.168.1.191; 192.168.1.30
8,d4:dc:cd:b4:26:3e,2101,1,192.168.1.250
9,dc:56:e7:5b:61:41,2056,1,192.168.1.133


In [35]:
train_ip_map_df = device_map_df[
    (device_map_df["confidence"] == "high") &
    (device_map_df["ip"] != "192.168.1.1")
].copy()

display(train_ip_map_df)

,ip,best_mac,best_count,num_macs,total_count,best_ratio,confidence
0,192.168.1.152,00:0c:29:d2:b0:02,1710340,1,1710340,1.0,high
1,192.168.1.192,60:14:b3:b1:91:73,125095,1,125095,1.0,high
2,192.168.1.190,00:0c:29:ee:e0:7a,83448,1,83448,1.0,high
3,192.168.1.193,60:14:b3:b1:91:73,57810,1,57810,1.0,high
5,192.168.1.79,00:c3:f4:0f:67:73,31749,1,31749,1.0,high
6,192.168.1.30,80:3f:5d:10:17:e1,2581,1,2581,1.0,high
7,192.168.1.250,d4:dc:cd:b4:26:3e,2101,1,2101,1.0,high
8,192.168.1.133,dc:56:e7:5b:61:41,2056,1,2056,1.0,high
9,192.168.1.194,60:14:b3:b1:91:73,1334,1,1334,1.0,high
10,192.168.1.103,9c:b6:d0:8e:86:79,1096,1,1096,1.0,high


In [36]:
final_service_df = pd.DataFrame([
    {"service_name": "service_1", "best_mac": "00:0c:29:d2:b0:02", "ip_list": "192.168.1.152"},
    {"service_name": "service_2", "best_mac": "60:14:b3:b1:91:73", "ip_list": "192.168.1.192; 192.168.1.193; 192.168.1.194; 192.168.1.146"},
    {"service_name": "service_3", "best_mac": "00:0c:29:ee:e0:7a", "ip_list": "192.168.1.190"},
    {"service_name": "service_4", "best_mac": "00:c3:f4:0f:67:73", "ip_list": "192.168.1.79"},
    {"service_name": "service_5", "best_mac": "80:3f:5d:10:17:e1", "ip_list": "192.168.1.17; 192.168.1.191; 192.168.1.30"},
    {"service_name": "service_6", "best_mac": "d4:dc:cd:b4:26:3e", "ip_list": "192.168.1.250"},
    {"service_name": "service_7", "best_mac": "dc:56:e7:5b:61:41", "ip_list": "192.168.1.133"},
])

display(final_service_df)

,service_name,best_mac,ip_list
0,service_1,00:0c:29:d2:b0:02,192.168.1.152
1,service_2,60:14:b3:b1:91:73,192.168.1.192; 192.168.1.193; 192.168.1.194; 1...
2,service_3,00:0c:29:ee:e0:7a,192.168.1.190
3,service_4,00:c3:f4:0f:67:73,192.168.1.79
4,service_5,80:3f:5d:10:17:e1,192.168.1.17; 192.168.1.191; 192.168.1.30
5,service_6,d4:dc:cd:b4:26:3e,192.168.1.250
6,service_7,dc:56:e7:5b:61:41,192.168.1.133


In [37]:
service_rows = []

for _, row in final_service_df.iterrows():
    for ip_addr in row["ip_list"].split("; "):
        service_rows.append({
            "ip": ip_addr.strip(),
            "best_mac": row["best_mac"],
            "service_name": row["service_name"]
        })

final_ip_map_df = pd.DataFrame(service_rows)
final_ip_map_df = final_ip_map_df.sort_values("ip").reset_index(drop=True)

display(final_ip_map_df)

,ip,best_mac,service_name
0,192.168.1.133,dc:56:e7:5b:61:41,service_7
1,192.168.1.146,60:14:b3:b1:91:73,service_2
2,192.168.1.152,00:0c:29:d2:b0:02,service_1
3,192.168.1.17,80:3f:5d:10:17:e1,service_5
4,192.168.1.190,00:0c:29:ee:e0:7a,service_3
5,192.168.1.191,80:3f:5d:10:17:e1,service_5
6,192.168.1.192,60:14:b3:b1:91:73,service_2
7,192.168.1.193,60:14:b3:b1:91:73,service_2
8,192.168.1.194,60:14:b3:b1:91:73,service_2
9,192.168.1.250,d4:dc:cd:b4:26:3e,service_6


In [38]:
flow_df_2 = count_ip_flows(pcap2)
flow_df_3 = count_ip_flows(pcap3)

display(flow_df_2.head(20))
display(flow_df_3.head(20))

print("flow_df_2:", flow_df_2.shape)
print("flow_df_3:", flow_df_3.shape)

,src_ip,dst_ip,src_port,dst_port,proto,packet_count
0,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,404490
1,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,404484
2,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,337735
3,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,313943
4,192.168.1.152,192.168.1.152,53972.0,10502.0,TCP,211854
5,192.168.1.152,192.168.1.195,1880.0,54163.0,TCP,211000
6,192.168.1.195,192.168.1.152,54163.0,1880.0,TCP,191507
7,192.168.1.152,192.168.1.195,1880.0,52786.0,TCP,146497
8,192.168.1.195,192.168.1.152,52786.0,1880.0,TCP,133764
9,192.168.1.152,192.168.1.152,10502.0,53972.0,TCP,105928


,src_ip,dst_ip,src_port,dst_port,proto,packet_count
0,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,267447
1,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,252181
2,192.168.1.152,192.168.1.152,34296.0,10502.0,TCP,201203
3,192.168.1.152,192.168.1.195,1880.0,49773.0,TCP,190061
4,192.168.1.195,192.168.1.152,49773.0,1880.0,TCP,174500
5,192.168.1.152,192.168.1.152,1880.0,51782.0,TCP,145534
6,192.168.1.152,192.168.1.152,51782.0,1880.0,TCP,145364
7,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,133628
8,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,133619
9,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,133521


flow_df_2: (14570, 6)
flow_df_3: (15306, 6)


In [39]:
ip_to_service = dict(zip(final_ip_map_df["ip"], final_ip_map_df["service_name"]))
ip_to_mac = dict(zip(final_ip_map_df["ip"], final_ip_map_df["best_mac"]))

labeled_flow_2 = add_service_labels(flow_df_2, ip_to_service, ip_to_mac)
labeled_flow_3 = add_service_labels(flow_df_3, ip_to_service, ip_to_mac)

train_flow_2 = labeled_flow_2[labeled_flow_2["is_labeled"]].copy()
train_flow_3 = labeled_flow_3[labeled_flow_3["is_labeled"]].copy()

display(train_flow_2.head(20))
display(train_flow_3.head(20))

print(train_flow_2["service_name"].value_counts())
print(train_flow_3["service_name"].value_counts())

,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,service_name,service_mac,direction,is_labeled
0,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,404490,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
1,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,404484,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
2,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,337735,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
3,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,313943,192.168.1.152,service_1,00:0c:29:d2:b0:02,in,True
4,192.168.1.152,192.168.1.152,53972.0,10502.0,TCP,211854,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
5,192.168.1.152,192.168.1.195,1880.0,54163.0,TCP,211000,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
7,192.168.1.152,192.168.1.195,1880.0,52786.0,TCP,146497,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
9,192.168.1.152,192.168.1.152,10502.0,53972.0,TCP,105928,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
10,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,47120,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
12,192.168.1.250,224.0.0.251,5353.0,5353.0,UDP,1069,192.168.1.250,service_6,d4:dc:cd:b4:26:3e,out,True


,src_ip,dst_ip,src_port,dst_port,proto,packet_count,local_ip,service_name,service_mac,direction,is_labeled
0,192.168.1.152,3.122.49.24,52976.0,1883.0,TCP,267447,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
1,3.122.49.24,192.168.1.152,1883.0,52976.0,TCP,252181,192.168.1.152,service_1,00:0c:29:d2:b0:02,in,True
2,192.168.1.152,192.168.1.152,34296.0,10502.0,TCP,201203,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
3,192.168.1.152,192.168.1.195,1880.0,49773.0,TCP,190061,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
5,192.168.1.152,192.168.1.152,1880.0,51782.0,TCP,145534,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
6,192.168.1.152,192.168.1.152,51782.0,1880.0,TCP,145364,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
7,192.168.1.152,192.168.1.195,1880.0,51323.0,TCP,133628,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
8,192.168.1.152,192.168.1.152,1880.0,40688.0,TCP,133619,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
9,192.168.1.152,192.168.1.152,40688.0,1880.0,TCP,133521,192.168.1.152,service_1,00:0c:29:d2:b0:02,out,True
10,192.168.1.192,192.168.1.152,40571.0,1880.0,TCP,121974,192.168.1.192,service_2,60:14:b3:b1:91:73,out,True


service_name
service_1    5761
service_3    5306
service_4    3466
service_2       5
service_5       3
service_7       2
service_6       2
Name: count, dtype: int64
service_name
service_1    6213
service_3    5030
service_4    3940
service_5      45
service_2      21
service_6       2
service_7       2
Name: count, dtype: int64
